# GTAN temporal folds — reference nodes + UID-disjoint-style edge attributes

This notebook is copied from the full historical-label GTAN temporal notebook.
The graph construction and edge attributes are kept the same, but the node-label
input protocol is changed:

- only a fixed subset of training UIDs is revealed as reference nodes;
- non-reference training nodes are supervised query nodes;
- validation nodes remain masked;
- downstream GTAN embeddings use a reference-safe extraction pass so a row's own
  target label is not embedded directly into its CNN/LSTM feature vector.


In [ ]:
!pip install -q torch_geometric

In [ ]:
import torch
TORCH = torch.__version__
!pip install -q pyg-lib torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-{TORCH}.html

In [ ]:
import torch_scatter, torch_sparse
print('GNN backend ready')

## GTAN module

The original long GTAN/XGBoost helper cell from the source notebook was disabled in this matched-edge version. The active GTAN implementation is defined below in the reference-node temporal cell.

Original helper cell disabled to keep Run All focused on the matched temporal GTAN experiment.

## Config + data

In [ ]:
# ===== Imports + config =====
import os, gc, math, time, copy, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.nn import TransformerConv
from torch_geometric.loader import NeighborLoader
from torch_geometric.data import Data
from sklearn.metrics import roc_auc_score, average_precision_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ---- window-level temporal-encoder config (one prediction per window) ----
WINDOW=20; BATCH=1024; EPOCHS=30; LR=1e-3; WEIGHT_DECAY=2e-4
GRAD_CLIP=1.0; EARLY_STOP_PATIENCE=6; SEED=42; N_SEEDS=3
HIDDEN_DIM=128; NUM_LAYERS=2; DROPOUT=0.3
USE_POS_WEIGHT=False; USE_STATIC_TOWER=True
CNN_CHANNELS=128; N_RESBLOCKS=3; N_ATTN_LAYERS=2; N_HEADS=4; ATTN_DROPOUT=0.2
STATIC_HIDDEN=64; USE_CLS=True; USE_MAX_POOL=True

# ---- GTAN graph: UID-disjoint-style edge representation ----
# Reference-node protocol: reveal only a fixed subset of training UIDs
# in each temporal fold. The rest of train nodes are supervised query nodes.
USE_REFERENCE_NODES = True
REFERENCE_BY_UID = True
REF_FRAC = 0.40
REF_SEED = 123
REFERENCE_OOF_FOLDS = 5
IDENTITY_COLS = ["uid", "card1_addr1", "card1_addr1_P_emaildomain", "DeviceInfo"]
USE_SIMILARITY_EDGES = True
EDGE_PER_TRANS = 12
K_SIM = 15
DECAY_TAUS = (1.0, 7.0, 30.0)
USE_COSINE = True
SIM_BACKEND = "gpu"
SIM_BATCH = 1024

GTAN_HIDDEN=32; GTAN_HEADS=4; GTAN_LAYERS=2; GTAN_DROP=0.2
GTAN_EPOCHS=30; GTAN_LR=3e-4; GTAN_WD=1e-5
GTAN_EARLY_STOP_PATIENCE=5; GTAN_EARLY_STOP_MIN_DELTA=1e-4
GTAN_BATCH=2048; GTAN_NEIGH=(16,10)
MIN_TRAIN_MONTHS=3

# Deterministic top-k pruning and weighted sampling mirror the UID-disjoint notebook.
PRUNE_TOPK = True
PRUNE_K = 15
USE_WEIGHTED_SAMPLING = True
USE_TEMPORAL_SAMPLING = False

# Train embeddings are extracted out-of-fold so CNN/LSTM train rows do not carry
# their own target label inside the GTAN embedding.
TRAIN_EMB_OOF_FOLDS = 5

torch.manual_seed(SEED); np.random.seed(SEED)


In [ ]:
# ===== Load data =====
# dataframe -> identity columns (for identity edges) + TransactionDT (timestamps)
df = pd.read_parquet("/kaggle/input/datasets/bachhoviet/parquets/X_train_copy4.parquet")
df.index = df.index.astype(np.int64)

TEMPORAL_SPLIT_DATA_DIR = "/kaggle/input/datasets/bachhoviet/split-data-npy"
X_seq = np.load(f"{TEMPORAL_SPLIT_DATA_DIR}/X_train_seq.npy")   # (N,20,234) float16, LEFT-padded
L_all = np.load(f"{TEMPORAL_SPLIT_DATA_DIR}/L_train.npy")
y_all = np.load(f"{TEMPORAL_SPLIT_DATA_DIR}/y_aligned.npy").astype(np.int64)
dtm   = np.load(f"{TEMPORAL_SPLIT_DATA_DIR}/dt_m_aligned.npy")
tids  = np.load(f"{TEMPORAL_SPLIT_DATA_DIR}/train_order.npy").astype(np.int64)
N, T, NF = X_seq.shape

X_node = np.asarray(X_seq[:, -1, :]).astype(np.float32)   # FULL 234 feats -> clustering + GNN
X_time = df.loc[tids, "TransactionDT"].to_numpy().astype(np.float64)
id_vals = {c: df.loc[tids, c].to_numpy() for c in IDENTITY_COLS}   # identity value per node
print("X_seq", X_seq.shape, "| GNN node feats:", X_node.shape[1],
      "| identity rels:", list(id_vals), "| months", sorted(np.unique(dtm).tolist()))


In [ ]:
# ===== Window dataset (casts to float32 per item so X_seq can stay float16) =====
class WindowDataset(Dataset):
    def __init__(self, X, lengths, y=None):
        self.X = X
        self.lengths = torch.from_numpy(lengths.astype("int64"))
        self.y = None if y is None else torch.from_numpy(y.astype("float32"))
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i):
        x = torch.from_numpy(np.asarray(self.X[i])).float()
        return (x, self.lengths[i]) if self.y is None else (x, self.lengths[i], self.y[i])

def make_loader(X, lengths, y, batch_size, shuffle):
    return DataLoader(WindowDataset(X, lengths, y), batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, pin_memory=False, drop_last=False)


## Per-fold GTAN embeddings (computed once, reused by CNN & LSTM)

In [ ]:
# ===== Expanding-fold runner (calls the CURRENT global train_one_fold) =====
def run_expanding_gtan_only(tag):
    oof = np.full(N, np.nan, dtype=np.float32)
    fold_aucs = []
    for vm in VAL_MONTHS:
        emb, tr_g, va_g = FOLD_EMB[vm]
        Xtr, Xva = augment_rows(tr_g, emb), augment_rows(va_g, emb)
        ytr, yva = y_all[tr_g].astype("float32"), y_all[va_g].astype("float32")
        Ltr, Lva = L_all[tr_g], L_all[va_g]
        nf = Xtr.shape[2]
        seed_preds = []
        for s in range(N_SEEDS):
            print(f"\n-- {tag} fold {vm} seed {SEED+s} --")
            vp, va, m = train_one_fold(
                Xtr, Ltr, ytr, Xva, Lva, yva,
                n_features=nf, epochs=EPOCHS, batch=BATCH, lr=LR,
                weight_decay=WEIGHT_DECAY, device=device,
                early_stop_patience=EARLY_STOP_PATIENCE,
                grad_clip=GRAD_CLIP, seed=SEED+s,
            )
            seed_preds.append(vp)
            del m; gc.collect()
            if device.type == "cuda": torch.cuda.empty_cache()
        fold_pred = np.mean(seed_preds, axis=0)
        oof[va_g] = fold_pred
        fa = roc_auc_score(yva, fold_pred)
        fold_aucs.append((int(vm), round(fa, 4)))
        print(f"  fold val={vm}: nf={nf} AUC={fa:.4f}")
        del Xtr, Xva; gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()
    valid = ~np.isnan(oof)
    auc = roc_auc_score(y_all[valid], oof[valid])
    print(f"=== {tag} OOF AUC = {auc:.4f} | per-fold {fold_aucs}\n")
    return oof


In [ ]:
# ===== Reference-node temporal GTAN with UID-disjoint-style 11-dim edge_attr =====
REL_INDEX = {
    "uid": 0,
    "card1_addr1": 1,
    "card1_addr1_P_emaildomain": 2,
    "DeviceInfo": 3,
    "similarity": 4,
}

def _valid_id(v):
    v = np.asarray(v)
    ok = v != -1
    if np.issubdtype(v.dtype, np.floating):
        ok &= ~np.isnan(v)
    return ok

def _chain_pairs(labels, timestamps, edge_per_trans, valid=None):
    positions = np.arange(len(labels), dtype=np.int64)
    groups = np.asarray(labels)
    times_local = np.asarray(timestamps)
    if valid is not None:
        positions = positions[valid]
        groups = groups[valid]
        times_local = times_local[valid]
    if len(positions) == 0:
        empty = np.array([], dtype=np.int64)
        return empty, empty
    order = np.argsort(times_local, kind="mergesort")
    frame = pd.DataFrame({"node": positions[order], "group": groups[order]})
    src_list, dst_list = [], []
    for _, group_frame in frame.groupby("group", sort=False):
        idx = group_frame["node"].to_numpy()
        group_len = len(idx)
        for step in range(1, edge_per_trans):
            if step >= group_len:
                break
            src_list.append(idx[:group_len-step])
            dst_list.append(idx[step:])
    if src_list:
        return np.concatenate(src_list), np.concatenate(dst_list)
    empty = np.array([], dtype=np.int64)
    return empty, empty

@torch.no_grad()
def knn_gpu(X_norm_np, timestamps, uid_values, k, dev, batch=1024):
    n = X_norm_np.shape[0]
    X_norm_t = torch.from_numpy(X_norm_np).to(dev)
    time_t = torch.tensor(timestamps, device=dev)
    uid_codes = torch.tensor(pd.factorize(uid_values)[0], device=dev)
    src_parts, dst_parts = [], []
    for start in range(0, n, batch):
        end = min(start + batch, n)
        sims = X_norm_t[start:end] @ X_norm_t.T
        query_time = time_t[start:end].unsqueeze(1)
        query_uid = uid_codes[start:end].unsqueeze(1)
        valid = (time_t.unsqueeze(0) < query_time) & (uid_codes.unsqueeze(0) != query_uid)
        sims = sims.masked_fill(~valid, float("-inf"))
        kk = min(k, n)
        top_values, top_indices = torch.topk(sims, kk, dim=1)
        mask = torch.isfinite(top_values)
        rows = torch.arange(end - start, device=dev).unsqueeze(1).expand(-1, kk)[mask] + start
        cols = top_indices[mask]
        dst_parts.append(rows.cpu())
        src_parts.append(cols.cpu())
        del sims, valid, top_values, top_indices
    if dev.type == "cuda":
        torch.cuda.empty_cache()
    if not src_parts:
        empty = np.array([], dtype=np.int64)
        return empty, empty
    return torch.cat(src_parts).numpy(), torch.cat(dst_parts).numpy()

def prune_topk_per_destination(edge_index, edge_attr, k):
    dst = edge_index[1].cpu().numpy()
    cosine = edge_attr[:, 10].cpu().numpy() if edge_attr.shape[1] >= 11 else np.zeros(edge_attr.shape[0], np.float32)
    strength = cosine + edge_attr[:, 9].cpu().numpy() + edge_attr[:, 1].cpu().numpy()
    is_self = edge_attr[:, 8].cpu().numpy() > 0.5
    frame = pd.DataFrame({"dst": dst, "strength": strength, "edge": np.arange(len(dst))})
    frame["rank"] = frame.groupby("dst", sort=False)["strength"].rank(method="first", ascending=False)
    keep = (frame["rank"].to_numpy() <= k) | is_self
    selected = frame["edge"].to_numpy()[keep]
    return edge_index[:, selected], edge_attr[selected]

def build_uidstyle_edges(features, identities, timestamps, fold_tag):
    n = features.shape[0]
    norm = np.linalg.norm(features, axis=1, keepdims=True)
    norm[norm == 0] = 1.0
    features_norm = (features / norm).astype(np.float32)
    parts = []

    for column in IDENTITY_COLS:
        relation_id = REL_INDEX[column]
        values = identities[column]
        src, dst = _chain_pairs(values, timestamps, EDGE_PER_TRANS, valid=_valid_id(values))
        if len(src):
            parts.append((src, dst, relation_id))
            print(f"    {column}: {len(src):,} edges")

    if USE_SIMILARITY_EDGES:
        src, dst = knn_gpu(features_norm, timestamps, identities["uid"], K_SIM, device, batch=SIM_BATCH)
        if len(src):
            parts.append((src, dst, REL_INDEX["similarity"]))
            print(f"    similarity(kNN k={K_SIM}): {len(src):,} edges")
    else:
        print("    similarity edges: DISABLED")

    if parts:
        src_all = np.concatenate([part[0] for part in parts])
        dst_all = np.concatenate([part[1] for part in parts])
        rel_all = np.concatenate([
            np.full(len(part[0]), part[2], dtype=np.int8)
            for part in parts
        ])
        frame = pd.DataFrame({"src": src_all, "dst": dst_all})
        for relation_id in range(5):
            frame[f"r{relation_id}"] = (rel_all == relation_id).astype(np.float32)
        grouped = frame.groupby(["src", "dst"], sort=False).max().reset_index()
        src = grouped["src"].to_numpy()
        dst = grouped["dst"].to_numpy()
        relation_flags = grouped[[f"r{i}" for i in range(5)]].to_numpy(np.float32)
        relation_count = relation_flags.sum(axis=1, keepdims=True)
        delta_days = ((timestamps[dst] - timestamps[src]).astype(np.float32)) / 86400.0
        recency = np.stack(
            [np.exp(-delta_days / float(tau)) for tau in DECAY_TAUS],
            axis=1,
        ).astype(np.float32)
        cosine = np.empty((len(src), 1), dtype=np.float32)
        chunk = 1_000_000
        for start in range(0, len(src), chunk):
            end = min(start + chunk, len(src))
            cosine[start:end, 0] = (
                features_norm[src[start:end]] * features_norm[dst[start:end]]
            ).sum(axis=1)
        edge_attr_np = np.concatenate(
            [
                recency,
                relation_flags,
                np.zeros((len(src), 1), dtype=np.float32),
                relation_count,
                cosine,
            ],
            axis=1,
        ).astype(np.float32)
        edge_index = torch.tensor(np.stack([src, dst]), dtype=torch.long)
        edge_attr = torch.tensor(edge_attr_np)
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 11), dtype=torch.float32)

    loops = torch.arange(n, dtype=torch.long)
    self_attr = torch.zeros((n, 11), dtype=torch.float32)
    self_attr[:, 0:3] = 1.0
    self_attr[:, 8] = 1.0
    self_attr[:, 10] = 1.0
    edge_index = torch.cat([edge_index, torch.stack([loops, loops])], dim=1)
    edge_attr = torch.cat([edge_attr, self_attr], dim=0)

    print(f"    total edges incl self-loops: {edge_index.shape[1]:,}; edge_dim={edge_attr.shape[1]}")
    return edge_index, edge_attr

def _make_y_input(y, known_idx, n_classes=2):
    y_input = torch.full((len(y),), n_classes, dtype=torch.long)
    if len(known_idx):
        idx = torch.as_tensor(known_idx, dtype=torch.long)
        y_input[idx] = y.cpu()[idx]
    return y_input

def select_reference_and_query(train_idx, uid_values, seed):
    """Select fixed reference nodes inside one temporal fold.

    Reference labels are visible to GTAN message passing. Query-train and
    validation nodes keep the unknown-label token.
    """
    train_idx = np.asarray(train_idx, dtype=np.int64)
    if not USE_REFERENCE_NODES or REF_FRAC <= 0:
        return np.array([], dtype=np.int64), train_idx

    rng = np.random.default_rng(seed)
    uid_array = np.asarray(uid_values)
    if REFERENCE_BY_UID:
        train_uids = uid_array[train_idx]
        valid_uid_mask = _valid_id(train_uids)
        candidate_uids = pd.unique(train_uids[valid_uid_mask])
        if len(candidate_uids):
            n_reference = min(
                len(candidate_uids), max(1, int(round(REF_FRAC * len(candidate_uids))))
            )
            reference_uids = rng.permutation(candidate_uids)[:n_reference]
            reference_mask = np.isin(train_uids, reference_uids)
            reference_idx = train_idx[reference_mask]
        else:
            n_reference = min(len(train_idx), max(1, int(round(REF_FRAC * len(train_idx)))))
            reference_idx = np.sort(rng.permutation(train_idx)[:n_reference])
    else:
        n_reference = min(len(train_idx), max(1, int(round(REF_FRAC * len(train_idx)))))
        reference_idx = np.sort(rng.permutation(train_idx)[:n_reference])

    query_idx = np.setdiff1d(train_idx, reference_idx, assume_unique=False)
    if len(query_idx) == 0:
        raise ValueError("Reference split leaves no query-training nodes")
    return np.asarray(reference_idx, dtype=np.int64), np.asarray(query_idx, dtype=np.int64)

class GTANEdge(nn.Module):
    def __init__(self, in_feats, hidden_dim=32, heads=4, n_layers=2,
                 n_classes=2, dropout=0.2, edge_dim=11):
        super().__init__()
        self.n_classes = n_classes
        width = hidden_dim * heads
        self.label_emb = nn.Embedding(n_classes + 1, in_feats, padding_idx=n_classes)
        self.feat_lin = nn.Linear(in_feats, width)
        self.label_lin = nn.Linear(in_feats, width)
        self.label_proc = nn.Sequential(
            nn.BatchNorm1d(width), nn.PReLU(), nn.Dropout(dropout),
            nn.Linear(width, in_feats),
        )
        self.input_drop = nn.Dropout(dropout)
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        dim = in_feats
        for _ in range(n_layers):
            self.convs.append(
                TransformerConv(
                    dim, hidden_dim, heads=heads, concat=True, beta=True,
                    dropout=dropout, edge_dim=edge_dim,
                )
            )
            self.norms.append(nn.LayerNorm(width))
            dim = width
        self.act = nn.PReLU()
        self.drop = nn.Dropout(dropout)
        self.emb_dim = dim
        self.head = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.PReLU(),
            nn.Dropout(dropout), nn.Linear(dim, n_classes),
        )

    def forward(self, x, edge_index, y_input, edge_attr=None):
        labels = self.input_drop(self.label_emb(y_input))
        hidden = x + self.label_proc(self.feat_lin(x) + self.label_lin(labels))
        for conv, norm in zip(self.convs, self.norms):
            hidden = conv(hidden, edge_index, edge_attr=edge_attr)
            hidden = self.drop(self.act(norm(hidden)))
        return self.head(hidden), hidden

def _loader(x, edge_index, edge_attr, sampling_weight, node_time,
            y_input, input_nodes, batch_size, neighbors, shuffle, y=None):
    data = Data(x=x.cpu().float(), edge_index=edge_index.cpu())
    data.edge_attr = edge_attr.cpu().float()
    if sampling_weight is not None:
        data.edge_weight = sampling_weight.cpu().float()
    if node_time is not None:
        data.time = node_time.cpu().long()
    data.yik = y_input.cpu().long()
    if y is not None:
        data.y = y.cpu().long()

    kwargs = {}
    if USE_WEIGHTED_SAMPLING:
        kwargs["weight_attr"] = "edge_weight"
    if USE_TEMPORAL_SAMPLING:
        kwargs["time_attr"] = "time"
    try:
        return NeighborLoader(
            data, num_neighbors=list(neighbors), input_nodes=torch.as_tensor(input_nodes),
            batch_size=batch_size, shuffle=shuffle, **kwargs,
        )
    except Exception as error:
        print(
            f"  [loader] {type(error).__name__} with {list(kwargs)}; "
            "falling back to uniform sampling"
        )
        return NeighborLoader(
            data, num_neighbors=list(neighbors), input_nodes=torch.as_tensor(input_nodes),
            batch_size=batch_size, shuffle=shuffle,
        )

def _mask_seed_labels(y_input_batch, seed_count):
    y_masked = y_input_batch.clone()
    y_masked[:seed_count] = 2
    return y_masked

def train_gtan_edge(x, edge_index, edge_attr, y, query_idx, val_idx,
                    reference_idx, node_time, in_feats, dev, verbose=False):
    y_cpu = y.cpu()
    query_idx = np.asarray(query_idx, dtype=np.int64)
    val_idx = np.asarray(val_idx, dtype=np.int64)
    reference_idx = np.asarray(reference_idx, dtype=np.int64)
    positives = float((y_cpu[torch.as_tensor(query_idx)] == 1).sum().clamp(min=1))
    negatives = float((y_cpu[torch.as_tensor(query_idx)] == 0).sum().clamp(min=1))
    class_weight = torch.tensor([1.0, float(np.sqrt(negatives / positives))], device=dev)

    sampling_weight = None
    if USE_WEIGHTED_SAMPLING:
        cosine = edge_attr[:, 10].clamp(min=0, max=1)
        sampling_weight = (edge_attr[:, 9] + edge_attr[:, 1] + cosine + 1e-3).float()
    loader_time = torch.as_tensor(node_time, dtype=torch.long) if USE_TEMPORAL_SAMPLING else None

    model = GTANEdge(
        in_feats, GTAN_HIDDEN, GTAN_HEADS, GTAN_LAYERS, 2, GTAN_DROP,
        edge_dim=edge_attr.shape[1],
    ).to(dev)
    optimizer = torch.optim.Adam(model.parameters(), lr=GTAN_LR, weight_decay=GTAN_WD)
    y_input = _make_y_input(y_cpu, reference_idx)

    train_loader = _loader(
        x, edge_index, edge_attr, sampling_weight, loader_time,
        y_input, query_idx, GTAN_BATCH, GTAN_NEIGH, True, y=y,
    )
    validation_loader = _loader(
        x, edge_index, edge_attr, sampling_weight, loader_time,
        y_input, val_idx, GTAN_BATCH, GTAN_NEIGH, False, y=y,
    )

    best_auc, best_state = -1.0, None
    bad_epochs = 0
    for epoch in range(1, GTAN_EPOCHS + 1):
        model.train()
        total_loss = batches = 0
        for batch in train_loader:
            seed_count = batch.batch_size
            batch = batch.to(dev)
            batch_y_input = _mask_seed_labels(batch.yik, seed_count)
            optimizer.zero_grad()
            logits, _ = model(batch.x, batch.edge_index, batch_y_input, batch.edge_attr)
            loss = F.cross_entropy(logits[:seed_count], batch.y[:seed_count], weight=class_weight)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            optimizer.step()
            total_loss += float(loss)
            batches += 1

        model.eval()
        validation_prediction = np.zeros(y.shape[0], np.float32)
        with torch.no_grad():
            for batch in validation_loader:
                seed_count = batch.batch_size
                batch = batch.to(dev)
                logits, _ = model(batch.x, batch.edge_index, batch.yik, batch.edge_attr)
                node_ids = batch.n_id[:seed_count].cpu().numpy()
                validation_prediction[node_ids] = F.softmax(logits[:seed_count], dim=1)[:, 1].cpu().numpy()
        val_auc = roc_auc_score(y_cpu.numpy()[val_idx], validation_prediction[val_idx])
        improved = val_auc > best_auc + GTAN_EARLY_STOP_MIN_DELTA
        if improved:
            best_auc = val_auc
            best_state = {key: value.detach().clone() for key, value in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
        if verbose:
            print(
                f"   ep {epoch:2d}/{GTAN_EPOCHS} "
                f"loss={total_loss/max(batches,1):.4f} "
                f"val_auc={val_auc:.4f} best={best_auc:.4f} bad={bad_epochs}"
            )
        if GTAN_EARLY_STOP_PATIENCE and bad_epochs >= GTAN_EARLY_STOP_PATIENCE:
            print(
                f"   early stop GTAN at epoch {epoch} "
                f"(best_val_auc={best_auc:.4f})"
            )
            break

    if best_state:
        model.load_state_dict(best_state)
    print(f"   >>> GTAN best val AUC = {best_auc:.4f}")
    return model, y_input, sampling_weight, loader_time

@torch.no_grad()
def _predict(model, x, edge_index, edge_attr, sampling_weight, node_time, y_input, input_nodes, dev):
    model.eval()
    input_nodes = np.asarray(input_nodes, dtype=np.int64)
    probability = np.zeros(x.shape[0], np.float32)
    embedding = np.zeros((x.shape[0], model.emb_dim), np.float32)
    loader = _loader(
        x, edge_index, edge_attr, sampling_weight, node_time,
        y_input, input_nodes, GTAN_BATCH, GTAN_NEIGH, False,
    )
    for batch in loader:
        seed_count = batch.batch_size
        batch = batch.to(dev)
        logits, hidden = model(batch.x, batch.edge_index, batch.yik, batch.edge_attr)
        node_ids = batch.n_id[:seed_count].cpu().numpy()
        probability[node_ids] = F.softmax(logits[:seed_count], dim=1)[:, 1].cpu().numpy()
        embedding[node_ids] = hidden[:seed_count].cpu().numpy()
    return probability, embedding

def reference_safe_embeddings(model, x, edge_index, edge_attr, sampling_weight,
                              node_time, y, reference_idx, query_idx, val_idx, dev):
    """Build downstream embeddings without exposing a row's own target label."""
    n_nodes = x.shape[0]
    probability = np.zeros(n_nodes, np.float32)
    embedding = np.zeros((n_nodes, model.emb_dim), np.float32)
    reference_idx = np.asarray(reference_idx, dtype=np.int64)
    query_idx = np.asarray(query_idx, dtype=np.int64)
    val_idx = np.asarray(val_idx, dtype=np.int64)

    base_y_input = _make_y_input(y.cpu(), reference_idx)
    ordinary_nodes = np.concatenate([query_idx, val_idx])
    pred, emb = _predict(
        model, x, edge_index, edge_attr, sampling_weight, node_time,
        base_y_input, ordinary_nodes, dev,
    )
    probability[ordinary_nodes] = pred[ordinary_nodes]
    embedding[ordinary_nodes] = emb[ordinary_nodes]

    if len(reference_idx):
        rng = np.random.default_rng(REF_SEED)
        shuffled = rng.permutation(reference_idx)
        for held_reference in np.array_split(shuffled, REFERENCE_OOF_FOLDS):
            if len(held_reference) == 0:
                continue
            revealed_reference = np.setdiff1d(reference_idx, held_reference, assume_unique=False)
            held_y_input = _make_y_input(y.cpu(), revealed_reference)
            pred, emb = _predict(
                model, x, edge_index, edge_attr, sampling_weight, node_time,
                held_y_input, held_reference, dev,
            )
            probability[held_reference] = pred[held_reference]
            embedding[held_reference] = emb[held_reference]
    return probability, embedding

def gtan_emb_for_fold(vm, verbose=False):
    sub = np.flatnonzero(dtm <= vm)
    train_local = np.flatnonzero(dtm[sub] < vm)
    val_local = np.flatnonzero(dtm[sub] == vm)
    features = X_node[sub].astype(np.float32, copy=False)
    timestamps = X_time[sub]
    identities = {column: id_vals[column][sub] for column in IDENTITY_COLS}

    cache_tag = (
        f"temporal_uidstyle_vm{int(vm)}_rel{'-'.join(IDENTITY_COLS)}_"
        f"sim{int(USE_SIMILARITY_EDGES)}_k{K_SIM}_ept{EDGE_PER_TRANS}_"
        f"cos{int(USE_COSINE)}_{SIM_BACKEND}"
    )
    edge_path = f"/kaggle/working/edge_index_{cache_tag}.npy"
    attr_path = f"/kaggle/working/edge_attr_{cache_tag}.npy"
    if os.path.exists(edge_path) and os.path.exists(attr_path):
        edge_index = torch.from_numpy(np.load(edge_path))
        raw_edge_attr = torch.from_numpy(np.load(attr_path))
        print(f"    loaded cached graph: {edge_index.shape[1]:,} edges, edge_dim={raw_edge_attr.shape[1]}")
    else:
        edge_index, raw_edge_attr = build_uidstyle_edges(features, identities, timestamps, cache_tag)
        np.save(edge_path, edge_index.numpy())
        np.save(attr_path, raw_edge_attr.numpy())

    if PRUNE_TOPK:
        original_edges = edge_index.shape[1]
        edge_index, raw_edge_attr = prune_topk_per_destination(edge_index, raw_edge_attr, PRUNE_K)
        print(f"    top-{PRUNE_K} prune: {original_edges:,} -> {edge_index.shape[1]:,} edges")

    reference_local, query_local = select_reference_and_query(
        train_local, identities["uid"], REF_SEED + int(vm)
    )
    print(
        f"    fold graph: nodes={len(sub):,}, train={len(train_local):,}, "
        f"reference={len(reference_local):,}, query={len(query_local):,}, "
        f"val={len(val_local):,}, edge_dim={raw_edge_attr.shape[1]}"
    )
    print(
        "    label protocol: reference train labels are known; "
        "query-train and validation labels are masked"
    )

    x = torch.tensor(features)
    y_tensor = torch.tensor(y_all[sub], dtype=torch.long)
    model, y_input, sampling_weight, loader_time = train_gtan_edge(
        x, edge_index, raw_edge_attr, y_tensor, query_local, val_local,
        reference_local, timestamps, features.shape[1], device, verbose=verbose,
    )
    model_device = next(model.parameters()).device

    probability, local_embedding = reference_safe_embeddings(
        model, x, edge_index, raw_edge_attr, sampling_weight, loader_time,
        y_tensor, reference_local, query_local, val_local, model_device,
    )
    val_prediction = probability[val_local].astype(np.float32, copy=True)
    gtan_auc = roc_auc_score(y_all[sub][val_local], val_prediction)

    train_global = sub[train_local]
    val_global = sub[val_local]
    embedding = np.zeros((N, local_embedding.shape[1]), dtype=np.float32)
    embedding[sub] = local_embedding
    mean = embedding[train_global].mean(axis=0)
    std = embedding[train_global].std(axis=0) + 1e-6
    embedding = ((embedding - mean) / std).astype(np.float32)

    del model
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return embedding, train_global, val_global, val_prediction, gtan_auc

def augment_rows(rows, embedding):
    row_sequences = np.asarray(X_seq[rows])
    n_rows = len(rows)
    emb_dim = embedding.shape[1]
    repeated_embedding = np.zeros((n_rows, T, emb_dim), dtype=np.float16)
    lengths = L_all[rows].astype(int)
    for position, global_idx in enumerate(rows):
        repeated_embedding[position, T-lengths[position]:, :] = embedding[global_idx]
    return np.concatenate([row_sequences, repeated_embedding], axis=2)

months = sorted(np.unique(dtm).tolist())
VAL_MONTHS = months[MIN_TRAIN_MONTHS:]
FOLD_EMB = {}
GTAN_OOF = np.full(N, np.nan, dtype=np.float32)
GTAN_FOLD_AUCS = []

for vm in VAL_MONTHS:
    print(f"### GTAN reference-node uid-style-edge fold: train <{vm} -> validate {vm} ###")
    emb, train_rows, val_rows, val_prediction, gtan_auc = gtan_emb_for_fold(vm, verbose=True)
    FOLD_EMB[vm] = (emb, train_rows, val_rows)
    GTAN_OOF[val_rows] = val_prediction
    GTAN_FOLD_AUCS.append((int(vm), float(gtan_auc)))
    print(f">>> GTAN val AUC (month {vm}) = {gtan_auc:.4f}\n")

gtan_valid = ~np.isnan(GTAN_OOF)
GTAN_OVERALL_OOF_AUC = roc_auc_score(y_all[gtan_valid], GTAN_OOF[gtan_valid])
GTAN_OVERALL_OOF_AP = average_precision_score(y_all[gtan_valid], GTAN_OOF[gtan_valid])

print("=" * 82)
print("GTAN-ALONE REFERENCE-NODE POOLED OOF SUMMARY")
print(f"Validated rows: {gtan_valid.sum():,} of {N:,} ({gtan_valid.mean():.1%})")
print(f"Overall pooled OOF ROC-AUC = {GTAN_OVERALL_OOF_AUC:.6f}")
print(f"Overall pooled OOF AP      = {GTAN_OVERALL_OOF_AP:.6f}")
print("Per-fold ROC-AUC:", [(month, round(auc, 6)) for month, auc in GTAN_FOLD_AUCS])
print("=" * 82)


## CNN + ResNet + Attention

In [ ]:
"""
Layout B v2 — Conv1D + ResNet1D + STACKED Self-Attention + CLS + Mean/Max Pool + Static Tower.

Upgrades over fraud_cnn_resnet_attention.py:
  (1) Masked Mean + Max pool concatenated     → captures average + spike behavior
  (2) Learnable CLS token                     → transformer-style classification readout
  (3) Static-row tower (last transaction MLP) → recovers XGBoost-style intra-row signal
  (3) Stack of N transformer encoder blocks   → multi-hop attention reasoning

Same forward signature as v1:
    forward(x, lengths) -> logits of shape (B, 1)

Pipeline:
    (B, T, F)
        ├── transpose ───────────────────────► (B, F, T)
        ├── Conv1D stem (F → C, k=3) ────────► (B, C, T)
        ├── ResBlock1D × N_RESBLOCKS ────────► (B, C, T)
        ├── transpose + positional emb ──────► (B, T, C)
        ├── [optionally prepend CLS token] ──► (B, T+1, C)
        ├── TransformerBlock × N_ATTN_LAYERS ► (B, T+1, C)
        ├── pooling:
        │     - mean over real timesteps     → (B, C)
        │     - max  over real timesteps     → (B, C)        [optional]
        │     - CLS readout                  → (B, C)        [optional]
        │     - static MLP on last row       → (B, S)        [optional]
        ├── concatenate the parts            ► (B, pool_dim)
        └── MLP head                         ► (B, 1)
"""

import torch
from torch import nn


# =============================================================================
# 1D Residual Block — shape preserving (unchanged from v1)
# =============================================================================
class ResBlock1D(nn.Module):
    def __init__(self, c: int, k: int = 3, drop: float = 0.1):
        super().__init__()
        p = k // 2
        self.conv1 = nn.Conv1d(c, c, kernel_size=k, padding=p)
        self.bn1   = nn.BatchNorm1d(c)
        self.conv2 = nn.Conv1d(c, c, kernel_size=k, padding=p)
        self.bn2   = nn.BatchNorm1d(c)
        self.drop  = nn.Dropout(drop)
        self.act   = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x                                   # SKIP BRANCH
        out = self.act(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        out = out + identity                           # SKIP CONNECTION
        return self.act(out)


# =============================================================================
# Transformer encoder block — one self-attention + FF, both with residual+LN
# =============================================================================
class TransformerBlock(nn.Module):
    def __init__(self, c: int, n_heads: int, drop: float = 0.2):
        super().__init__()
        self.attn  = nn.MultiheadAttention(
            embed_dim=c, num_heads=n_heads, dropout=drop, batch_first=True,
        )
        self.norm1 = nn.LayerNorm(c)
        self.ff    = nn.Sequential(
            nn.Linear(c, c * 2),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(c * 2, c),
        )
        self.norm2 = nn.LayerNorm(c)

    def forward(self, h: torch.Tensor, key_padding_mask: torch.Tensor) -> torch.Tensor:
        a, _ = self.attn(h, h, h,
                         key_padding_mask=key_padding_mask,
                         need_weights=False)
        h = self.norm1(h + a)
        h = self.norm2(h + self.ff(h))
        return h


# =============================================================================
# Full v2 model
# =============================================================================
class FraudCNNResAttnV2(nn.Module):
    """
    Args
    ----
    n_features        : number of features per timestep (e.g. 244)
    window            : sequence length T (e.g. 20)
    c_hidden          : channel width of Conv stem / ResNet / attention
    n_resblocks       : number of ResBlock1D in the CNN stack
    n_attn_layers     : number of TransformerBlock layers (was 1 in v1)        ◄── NEW
    n_heads           : self-attention heads (must divide c_hidden)
    drop              : dropout used in ResBlocks, attention, FF, head, static tower
    static_hidden     : hidden width of the static-row tower
    use_cls           : prepend a learnable CLS token, use its embedding for readout ◄── NEW
    use_max_pool      : concat masked-max pool with mean pool                      ◄── NEW
    use_static_tower  : run the last raw row through an MLP and concat it          ◄── NEW
    output_dim        : final logit dim (keep =1 for BCEWithLogitsLoss)
    """
    def __init__(self,
                 n_features: int,
                 window: int = 20,
                 c_hidden: int = 128,
                 n_resblocks: int = 3,
                 n_attn_layers: int = 2,
                 n_heads: int = 4,
                 drop: float = 0.2,
                 static_hidden: int = 64,
                 use_cls: bool = True,
                 use_max_pool: bool = True,
                 use_static_tower: bool = True,
                 output_dim: int = 1):
        super().__init__()
        assert c_hidden % n_heads == 0, "c_hidden must be divisible by n_heads"

        self.window           = window
        self.use_cls          = use_cls
        self.use_max_pool     = use_max_pool
        self.use_static_tower = use_static_tower

        # ------- Stage 1 — Conv1D stem -------
        self.stem = nn.Sequential(
            nn.Conv1d(n_features, c_hidden, kernel_size=3, padding=1),
            nn.BatchNorm1d(c_hidden),
            nn.ReLU(),
        )

        # ------- Stage 2 — ResNet1D stack -------
        self.resblocks = nn.Sequential(*[
            ResBlock1D(c_hidden, k=3, drop=drop)
            for _ in range(n_resblocks)
        ])

        # ------- Stage 3 — positional embedding (over the WINDOW positions only;
        #                  the CLS token is added on top after this)
        self.pos = nn.Parameter(torch.zeros(1, window, c_hidden))
        nn.init.trunc_normal_(self.pos, std=0.02)

        # ------- (Optional) CLS token -------
        if use_cls:
            self.cls = nn.Parameter(torch.zeros(1, 1, c_hidden))
            nn.init.trunc_normal_(self.cls, std=0.02)

        # ------- Stage 4 — STACK of Transformer blocks -------
        self.attn_blocks = nn.ModuleList([
            TransformerBlock(c_hidden, n_heads, drop=drop)
            for _ in range(n_attn_layers)
        ])

        # ------- (Optional) Static-row tower -------
        if use_static_tower:
            self.static_mlp = nn.Sequential(
                nn.Linear(n_features, 256), nn.ReLU(), nn.Dropout(drop),
                nn.Linear(256, static_hidden), nn.ReLU(),
            )

        # ------- Compute pool dim from the toggles -------
        pool_dim = c_hidden                       # mean is always present
        if use_max_pool:    pool_dim += c_hidden
        if use_cls:         pool_dim += c_hidden
        if use_static_tower:pool_dim += static_hidden

        # ------- Stage 5 — Head -------
        self.head = nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(pool_dim, 64), nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(64, output_dim),
        )

    # ----------------------------------------------------------------------
    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        # x: (B, T, F)  left-padded     lengths: (B,) real-step count
        B, T, F = x.shape

        # Keep last (most-recent) raw row aside for the static tower.
        # Left-padding writes to the RIGHT end, so x[:, -1, :] is always real.
        last_row = x[:, -1, :]                           # (B, F)

        # ----- CNN path (B, T, F) → (B, T, C) -----
        x_t = x.transpose(1, 2)                          # (B, F, T)
        h   = self.stem(x_t)                             # (B, C, T)
        h   = self.resblocks(h)                          # (B, C, T)
        h   = h.transpose(1, 2)                          # (B, T, C)
        h   = h + self.pos                               # positional encoding (over T only)

        # ----- Build real-position mask from `lengths` -----
        idx          = torch.arange(T, device=h.device).unsqueeze(0)   # (1, T)
        pos_from_end = T - 1 - idx                                     # (1, T)
        real         = pos_from_end < lengths.unsqueeze(1)             # (B, T) — True = real

        # ----- (Optional) Prepend CLS token -----
        if self.use_cls:
            cls = self.cls.expand(B, -1, -1)             # (B, 1, C)
            h   = torch.cat([cls, h], dim=1)             # (B, T+1, C)
            # CLS is always "real" so attention won't mask it
            cls_real = torch.ones(B, 1, dtype=torch.bool, device=h.device)
            real_full = torch.cat([cls_real, real], dim=1)              # (B, T+1)
            key_padding_mask = ~real_full
        else:
            key_padding_mask = ~real

        # ----- Stacked Transformer blocks -----
        for blk in self.attn_blocks:
            h = blk(h, key_padding_mask)

        # ----- Split CLS embedding from the time tokens -----
        if self.use_cls:
            cls_out = h[:, 0, :]                         # (B, C)
            seq_h   = h[:, 1:, :]                        # (B, T, C)
        else:
            seq_h = h                                    # (B, T, C)

        # ----- Pool over REAL timesteps only -----
        mask_f = real.unsqueeze(-1).float()              # (B, T, 1)
        cnt    = mask_f.sum(dim=1).clamp(min=1.0)        # (B, 1)

        pooled_parts = [(seq_h * mask_f).sum(dim=1) / cnt]   # mean

        if self.use_max_pool:
            # mask out padded positions with -inf so they never win the max
            seq_h_for_max = seq_h.masked_fill(~real.unsqueeze(-1), float('-inf'))
            pooled_parts.append(seq_h_for_max.max(dim=1).values)

        if self.use_cls:
            pooled_parts.append(cls_out)

        if self.use_static_tower:
            pooled_parts.append(self.static_mlp(last_row))

        pooled = torch.cat(pooled_parts, dim=1)          # (B, pool_dim)
        return self.head(pooled)                         # (B, 1)


class FraudCNNResAttnV2PerStep(nn.Module):
    """
    Per-timestep CNN/ResNet/Attention model.

    Input : X shape (B, T, F), left-padded
            lengths shape (B,)
    Output: logits shape (B, T)

    Train with masked BCE:
        loss = BCE(logits[M], Y[M])
    """
    def __init__(self,
                 n_features: int,
                 window: int = 20,
                 c_hidden: int = 128,
                 n_resblocks: int = 3,
                 n_attn_layers: int = 2,
                 n_heads: int = 4,
                 drop: float = 0.2):
        super().__init__()

        assert c_hidden % n_heads == 0

        self.window = window

        self.stem = nn.Sequential(
            nn.Conv1d(n_features, c_hidden, kernel_size=3, padding=1),
            nn.BatchNorm1d(c_hidden),
            nn.ReLU(),
        )

        self.resblocks = nn.Sequential(*[
            ResBlock1D(c_hidden, k=3, drop=drop)
            for _ in range(n_resblocks)
        ])

        self.pos = nn.Parameter(torch.zeros(1, window, c_hidden))
        nn.init.trunc_normal_(self.pos, std=0.02)

        self.attn_blocks = nn.ModuleList([
            TransformerBlock(c_hidden, n_heads, drop=drop)
            for _ in range(n_attn_layers)
        ])

        # One logit per timestep
        self.token_head = nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(c_hidden, 64),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(64, 1),
        )

    def forward(self, x, lengths):
        # x: (B, T, F), LEFT-padded
        B, T, _ = x.shape

        h = x.transpose(1, 2)        # (B, F, T)
        h = self.stem(h)             # (B, C, T)
        h = self.resblocks(h)        # (B, C, T)
        h = h.transpose(1, 2)        # (B, T, C)

        h = h + self.pos[:, :T, :]

        # left-padding mask: real positions are at the RIGHT end
        idx = torch.arange(T, device=x.device).unsqueeze(0)
        real = (T - 1 - idx) < lengths.to(x.device).unsqueeze(1)
        key_padding_mask = ~real

        for blk in self.attn_blocks:
            h = blk(h, key_padding_mask)

        logits = self.token_head(h).squeeze(-1)   # (B, T)
        return logits

In [ ]:
def train_one_fold(X_tr, L_tr, y_tr, X_va, L_va, y_va, n_features,
                   epochs, batch, lr, weight_decay, device,
                   early_stop_patience, grad_clip, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = FraudCNNResAttnV2(
        n_features,
        c_hidden=CNN_CHANNELS,
        n_resblocks=N_RESBLOCKS,
        n_attn_layers=N_ATTN_LAYERS,
        n_heads=N_HEADS,
        drop=ATTN_DROPOUT,
        static_hidden=STATIC_HIDDEN,
        use_cls=USE_CLS,
        use_max_pool=USE_MAX_POOL,
        use_static_tower=USE_STATIC_TOWER,
    ).to(device) # CNN_ResNet_Attention v2
    if USE_POS_WEIGHT:
        pos = float((y_tr == 1).sum()); neg = float(len(y_tr) - pos)
        pw = torch.tensor([np.sqrt(neg / max(pos, 1.0))], device=device, dtype=torch.float32)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)
    else:
        loss_fn = nn.BCEWithLogitsLoss()                            # plain BCE → better AUC

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    train_loader = make_loader(X_tr, L_tr, y_tr, batch_size=batch, shuffle=True)
    val_loader   = make_loader(X_va, L_va, y_va, batch_size=batch, shuffle=False)

    steps = max(1, math.ceil(len(X_tr) / batch))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=steps,
        pct_start=0.1, anneal_strategy='cos',
    )

    best_auc, best_state, best_val_preds, bad = -1.0, None, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time(); running, n_seen = 0.0, 0
        for xb, lb, yb in train_loader:
            xb = xb.to(device); lb = lb.to(device); yb = yb.to(device)
            optimizer.zero_grad()
            logits = logits = model(xb, lb).reshape(-1) 
            loss = loss_fn(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step(); scheduler.step()
            running += loss.item() * xb.size(0); n_seen += xb.size(0)
        train_loss = running / max(n_seen, 1)

        model.eval(); preds = []
        with torch.no_grad():
            for xb, lb, _ in val_loader:
                xb = xb.to(device); lb = lb.to(device)
                preds.append(torch.sigmoid(model(xb, lb).reshape(-1)).cpu().numpy())
        val_preds = np.concatenate(preds)
        val_auc = roc_auc_score(y_va, val_preds)

        print(f'   ep {epoch:>2}/{epochs}  loss={train_loss:.4f}  val_auc={val_auc:.4f}  ({time.time()-t0:.1f}s)')
        if val_auc > best_auc:
            best_auc = val_auc; best_state = copy.deepcopy(model.state_dict())
            best_val_preds = val_preds; bad = 0
        else:
            bad += 1
            if bad >= early_stop_patience:
                print(f'   early stop at epoch {epoch}'); break

    if best_state is not None: model.load_state_dict(best_state)
    return best_val_preds, best_auc, model

In [ ]:
# CNN+ResNet+Attention + GTAN only.
# Baseline is intentionally skipped in this matched-edge temporal notebook.
cnn_gtan = run_expanding_gtan_only("CNN + GTAN")


## LSTM

In [ ]:
class FraudLSTM(nn.Module):
    def __init__(self, n_features, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS,
                 drop_prob=DROPOUT, output_dim=1,
                 use_static_tower=USE_STATIC_TOWER,
                 bidirectional=True):                     # NEW
        super().__init__()
        self.use_static_tower = use_static_tower
        self.bidirectional   = bidirectional             # NEW

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=drop_prob if num_layers > 1 else 0.0,
            bidirectional=bidirectional,                 # NEW
        )

        lstm_out_dim = hidden_dim * (2 if bidirectional else 1)   # NEW
        seq_out_dim  = 2 * lstm_out_dim                            # mean + max

        if use_static_tower:
            self.static_mlp = nn.Sequential(
                nn.Linear(n_features, 256), nn.ReLU(), nn.Dropout(drop_prob),
                nn.Linear(256, 128),        nn.ReLU(), nn.Dropout(drop_prob),
                nn.Linear(128, 64),         nn.ReLU(),
            )
            combined_dim = seq_out_dim + 64
        else:
            combined_dim = seq_out_dim

        self.head = nn.Sequential(
            nn.Linear(combined_dim, 64), nn.ReLU(), nn.Dropout(drop_prob),
            nn.Linear(64, output_dim),
        )

    def forward(self, x, lengths):
        # x: (B, T, F)  left-padded; lengths: (B,) real-step counts
        B, T, _ = x.shape
        lstm_out, _ = self.lstm(x)                                  # (B, T, H)

        # build a mask for the real (right-aligned) positions
        idx = torch.arange(T, device=x.device).unsqueeze(0)         # (1, T)
        pos_from_end = T - 1 - idx                                  # 0..T-1
        mask = pos_from_end < lengths.to(x.device).unsqueeze(1)     # (B, T)
        mask_f = mask.unsqueeze(-1).float()

        # masked mean
        sum_  = (lstm_out * mask_f).sum(dim=1)
        cnt   = mask_f.sum(dim=1).clamp(min=1.0)
        mean_pool = sum_ / cnt

        # masked max (pads → very negative)
        neg_inf  = torch.finfo(lstm_out.dtype).min
        max_pool = lstm_out.masked_fill(~mask.unsqueeze(-1), neg_inf).max(dim=1).values

        seq_vec = torch.cat([mean_pool, max_pool], dim=1)

        if self.use_static_tower:
            current  = x[:, -1, :]                                  # last real step
            stat_vec = self.static_mlp(current)
            feat = torch.cat([seq_vec, stat_vec], dim=1)
        else:
            feat = seq_vec

        return self.head(feat).squeeze(-1)

In [ ]:
def train_one_fold(X_tr, L_tr, y_tr, X_va, L_va, y_va, n_features,
                   epochs, batch, lr, weight_decay, device,
                   early_stop_patience, grad_clip, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = FraudLSTM(n_features).to(device)

    if USE_POS_WEIGHT:
        pos = float((y_tr == 1).sum()); neg = float(len(y_tr) - pos)
        pw = torch.tensor([np.sqrt(neg / max(pos, 1.0))], device=device, dtype=torch.float32)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)
    else:
        loss_fn = nn.BCEWithLogitsLoss()                            # plain BCE → better AUC

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    train_loader = make_loader(X_tr, L_tr, y_tr, batch_size=batch, shuffle=True)
    val_loader   = make_loader(X_va, L_va, y_va, batch_size=batch, shuffle=False)

    steps = max(1, math.ceil(len(X_tr) / batch))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=steps,
        pct_start=0.1, anneal_strategy='cos',
    )

    best_auc, best_state, best_val_preds, bad = -1.0, None, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time(); running, n_seen = 0.0, 0
        for xb, lb, yb in train_loader:
            xb = xb.to(device); lb = lb.to(device); yb = yb.to(device)
            optimizer.zero_grad()
            logits = model(xb, lb)
            loss = loss_fn(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step(); scheduler.step()
            running += loss.item() * xb.size(0); n_seen += xb.size(0)
        train_loss = running / max(n_seen, 1)

        model.eval(); preds = []
        with torch.no_grad():
            for xb, lb, _ in val_loader:
                xb = xb.to(device); lb = lb.to(device)
                preds.append(torch.sigmoid(model(xb, lb)).cpu().numpy())
        val_preds = np.concatenate(preds)
        val_auc = roc_auc_score(y_va, val_preds)

        print(f'   ep {epoch:>2}/{epochs}  loss={train_loss:.4f}  val_auc={val_auc:.4f}  ({time.time()-t0:.1f}s)')
        if val_auc > best_auc:
            best_auc = val_auc; best_state = copy.deepcopy(model.state_dict())
            best_val_preds = val_preds; bad = 0
        else:
            bad += 1
            if bad >= early_stop_patience:
                print(f'   early stop at epoch {epoch}'); break

    if best_state is not None: model.load_state_dict(best_state)
    return best_val_preds, best_auc, model

In [ ]:
# LSTM + GTAN only.
# Baseline is intentionally skipped in this matched-edge temporal notebook.
lstm_gtan = run_expanding_gtan_only("LSTM + GTAN")


## Summary

In [ ]:
print("=" * 82)
print("TEMPORAL FULL-TRAIN-LABEL GTAN WITH UID-DISJOINT-STYLE EDGE ATTRIBUTES")
print(f"GTAN alone pooled OOF AUC = {GTAN_OVERALL_OOF_AUC:.4f}")
for name, pred in [("CNN+GTAN", cnn_gtan), ("LSTM+GTAN", lstm_gtan)]:
    valid = ~np.isnan(pred)
    print(f"{name:10s} OOF AUC = {roc_auc_score(y_all[valid], pred[valid]):.4f}")
print("=" * 82)
